In [0]:
%python
from pyspark.sql.functions import expr,col,current_timestamp,current_date ,lit,sum , max , coalesce,row_number
from pyspark.sql.window import Window
from datetime import datetime
from pyspark.sql import DataFrame
from pyspark.sql.types import DecimalType , ShortType
import os

class OrderProcessor:
    def __init__(self):

        self.spark = spark
        self.checkpointlocation_txn = '**'

        self.process_dt = datetime.now()

    def read_staging_df(self) -> DataFrame:

        return (
            self.spark.readStream.table("dev.spark_db.order_stg")
        )
    
    def max_transaction_id(self):
        transaction_df = self.spark.read.table("dev.spark_db.transaction")

        return (
            transaction_df.select(coalesce(max(col("transaction_id")),lit(1000) ).alias("max_transaction_id") ).collect()[0]["max_transaction_id"]
        )

    def transaction_processor(self,stg_df:DataFrame) -> DataFrame:

        account = self.spark.read.table("dev.spark_db.account")
        asset = self.spark.read.table("dev.spark_db.asset_rate")

        max_txn_id = self.max_transaction_id()

        txn_df = (
            stg_df.alias("sd").join(account.alias("ac"), expr("sd.account_number = ac.account_number"), "inner")
                              .join(asset.alias("as") , 
                                    expr("""
                                         sd.asset_external_id = as.asset_external_id
                                         and 
                                         sd.as_of_dt = as.as_of_dt
                                         """),
                                    "inner")
                              .select( 
                                      expr(f"""
                                           (row_number() over(order by sd.order_stg_id) ) + {max_txn_id}
                                           """).alias("transaction_id"),
                                      "ac.account_number",
                                      "as.asset_external_id",
                                      "sd.order_type",
                                          expr("""
                                           case when sd.order_type = 'BUY' then
                                                    sd.quantity
                                                when sd.order_type = 'SELL' then
                                                    -(sd.quantity)
                                                end
                                           """)
                                      .cast(DecimalType(18,2)).alias("quantity"),
                                      col("sd.asset_rate").cast(DecimalType(18,2)),
                                          expr("""
                                           case when sd.order_type = 'BUY' then
                                                    -(sd.order_amt)
                                                when sd.order_type = 'SELL' then
                                                      sd.order_amt
                                                end
                                           """)
                                      .cast(DecimalType(18,2)).alias("order_amt"),
                                      col("sd.as_of_dt").alias("as_of_dt"),
                                      expr(" 1 as transaction_ver").cast(ShortType()),
                                      current_timestamp().alias("transaction_dml")
                                      )

        )
        return txn_df
    
    def write_to_delta_transaction(self,txn_df):

        (
            txn_df.write.format("delta")
                        .mode("append")
                        .saveAsTable("dev.spark_db.transaction")
        )

    def write_to_dynamodb_transaction(self,txn_df):

        def process_record(rows):
            
            import boto3

            dynamodb = boto3.resource(
                                "dynamodb",
                                region_name = "**",
                                aws_access_key_id = "**",
                                aws_secret_access_key = "**"
                            )
            
            table = dynamodb.Table('transaction')

            with table.batch_writer() as batch:
                for row in rows:
                    batch.put_item(
                        Item = {
                            'transaction_id' : row['transaction_id'],
                            'account_number' : row['account_number'],
                            'asset_external_id' : row['asset_external_id'],
                            'order_type': row['order_type'],
                            'quantity' : row['quantity'],
                            'asset_rate' : row['asset_rate'],
                            'order_amt' : row['order_amt'],
                            'as_of_dt' : str(row['as_of_dt']),
                            'transaction_ver' : row['transaction_ver'],
                            'transaction_dml' : str(row['transaction_dml'])
                        }
                    )


        txn_df.foreachPartition(process_record)

    def process(self,batch_df , batch_id):

        txn = self.transaction_processor(batch_df)

        self.write_to_delta_transaction(txn)

        return (self.write_to_dynamodb_transaction(txn))

    
    def write_to_transaction(self,read_stg):
        return (
            read_stg.writeStream.format("delta")
                              .queryName("Transaction_Query")
                              .foreachBatch(self.process)
                              .outputMode("append")
                              .option("checkpointLocation" ,self.checkpointlocation_txn)
                              .trigger( availableNow = True)
                              .start()
        )

    def main(self):

        read_stg = self.read_staging_df()

        transaction_query = self.write_to_transaction(read_stg )

        #trnsaction_query.awaitTermination()
        return transaction_query



if __name__ == '__main__':
    op = OrderProcessor()
    op.main()
                            


